# Preventra &mdash; Phase 2: Post-Discharge Risk Trend (MIMIC-IV)

### Read this before running

Phase 2 was specified as **weekly** post-discharge re-scoring. MIMIC-IV **cannot support that
cadence**, and this notebook measures exactly why rather than papering over it.

`omr` is the only outpatient table in the `hosp` module. It contains four measurement types
(Weight, Blood Pressure, BMI, Height) recorded **opportunistically when a patient happens to
attend a clinic** &mdash; not on a monitoring schedule. On the demo extract the median gap from
discharge to the first post-discharge reading was **127 days**, and only ~1.6% of discharges had
readings in even two distinct weeks of the 30-day window.

**Section 3 re-measures this on your full data and prints a go/no-go.** If it confirms the demo
finding, the notebook continues down the honest path:

> An **irregular-interval** post-discharge risk update &mdash; last observation carried forward,
> change since discharge baseline, and an explicit *staleness* feature &mdash; rather than a
> fabricated weekly series.

A true weekly cadence needs a source that actually samples weekly: All of Us wearables, or
Preventra's own telemetry once it is live.

## 0 &middot; Setup

In [ ]:
import os, gc, re, glob, json, warnings, time
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

# --- Kaggle resource envelope -------------------------------------------------
# CPU session: ~32 GB RAM | /kaggle/working ~20 GB (persistent)
#              /kaggle/temp ~60 GB (scratch)  | /kaggle/input read-only
#
# labevents.csv is 18.4 GB on disk. A naive read_csv would need roughly 3-5x
# that in RAM once parsed, so it is NEVER loaded whole anywhere in this
# notebook. Every table over ~500 MB is streamed in chunks, filtered down to
# the study cohort, aggregated, and discarded.
# -----------------------------------------------------------------------------

CHUNK_ROWS     = 2_000_000   # rows per chunk; lower this if you still hit memory pressure
COMPACT_EVERY  = 8           # re-aggregate partial results every N chunks
CACHE_DIR      = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def mem_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

def report(label, df):
    print(f"  {label:<34} rows={len(df):>12,}  cols={df.shape[1]:>3}  mem={mem_mb(df):>8.1f} MB")

def downcast(df):
    """Shrink numeric columns in place. Typically halves memory."""
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    return df

def cached(name):
    """Decorator-free cache helper: returns path for a parquet cache file."""
    return os.path.join(CACHE_DIR, f"{name}.parquet")

def load_cache(name):
    p = cached(name)
    if os.path.exists(p):
        print(f"  [cache hit] {name}")
        return pd.read_parquet(p)
    return None

def save_cache(df, name):
    df.to_parquet(cached(name), index=False)
    print(f"  [cached] {name}  ({os.path.getsize(cached(name))/1024**2:.1f} MB)")

In [ ]:
def find_mimic_root():
    """Locate the MIMIC-IV hosp directory on Kaggle or locally."""
    bases = ["/kaggle/input", "./data", ".", ".."]
    hits = []
    for base in bases:
        if not os.path.isdir(base):
            continue
        for depth in range(4):
            for stem in ("admissions.csv*", "sample_admissions.csv*"):
                pattern = os.path.join(base, *(["*"] * depth), stem)
                hits += [os.path.dirname(p) for p in glob.glob(pattern)]
    # Prefer a directory that also carries the other tables we need
    for h in sorted(set(hits), key=len):
        if glob.glob(os.path.join(h, "diagnoses_icd.csv*")) or \
           glob.glob(os.path.join(h, "sample_diagnoses_icd.csv*")):
            return h
    return hits[0] if hits else None

MIMIC = find_mimic_root()
if MIMIC is None:
    raise FileNotFoundError(
        "Could not find admissions.csv. On Kaggle, attach the MIMIC-IV dataset "
        "and it will mount under /kaggle/input/<slug>/."
    )
print("MIMIC-IV hosp directory:", MIMIC)

def tbl(name):
    """Resolve a table name to an actual path (.csv, .csv.gz, or demo sample_ prefix)."""
    for cand in (f"{name}.csv", f"{name}.csv.gz", f"sample_{name}.csv"):
        p = os.path.join(MIMIC, cand)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found in {MIMIC}")

def size_gb(name):
    try:
        return os.path.getsize(tbl(name)) / 1024**3
    except FileNotFoundError:
        return 0.0

print("\ntable sizes on disk:")
for t in ["admissions","patients","diagnoses_icd","drgcodes","procedures_icd",
          "omr","prescriptions","labevents","d_labitems"]:
    g = size_gb(t)
    tag = "  <-- STREAMED" if g > 0.5 else ""
    print(f"  {t:<18} {g:>8.2f} GB{tag}")

## 1 &middot; Cohort (same construction as Phase 1)

In [ ]:
# ---------------------------------------------------------------------------
# Cohort + 30-day unplanned readmission label
# ---------------------------------------------------------------------------
# Exclusions follow standard readmission-measure practice:
#   * died in hospital / discharged to hospice -> cannot be readmitted
#   * observation stays  -> not true inpatient admissions (CMS excludes them)
#   * elective returns   -> planned follow-up, not a care failure
#
# The "next admission" is computed on the FULL sequence BEFORE any rows are
# dropped. Filtering first would lose a readmission that happened to follow an
# excluded stay.
# ---------------------------------------------------------------------------

TERMINAL_DISPOSITIONS = {"DIED", "HOSPICE"}
OBSERVATION_TYPES = {"EU OBSERVATION", "OBSERVATION ADMIT",
                     "AMBULATORY OBSERVATION", "DIRECT OBSERVATION"}
READMIT_WINDOW_DAYS = 30

adm = pd.read_csv(
    tbl("admissions"),
    usecols=["subject_id","hadm_id","admittime","dischtime","admission_type",
             "admission_location","discharge_location","insurance","marital_status",
             "race","edregtime","edouttime","hospital_expire_flag"],
    parse_dates=["admittime","dischtime","edregtime","edouttime"],
)
patients = pd.read_csv(tbl("patients"), usecols=["subject_id","gender","anchor_age","anchor_year","dod"],
                  parse_dates=["dod"])
report("admissions", adm); report("patients", patients)

adm = adm.sort_values(["subject_id","admittime"]).reset_index(drop=True)

# --- prior utilisation, computed from the sequence itself --------------------
adm["prev_dischtime"] = adm.groupby("subject_id")["dischtime"].shift(1)
adm["n_prior_adm"] = adm.groupby("subject_id").cumcount()
adm["days_since_prev"] = (adm.admittime - adm.prev_dischtime).dt.total_seconds()/86400

# --- forward-looking label ---------------------------------------------------
adm["next_admittime"] = adm.groupby("subject_id")["admittime"].shift(-1)
adm["next_type"]      = adm.groupby("subject_id")["admission_type"].shift(-1)
adm["days_to_next"]   = (adm.next_admittime - adm.dischtime).dt.total_seconds()/86400

died        = adm.hospital_expire_flag.eq(1)
terminal    = adm.discharge_location.isin(TERMINAL_DISPOSITIONS)
observation = adm.admission_type.isin(OBSERVATION_TYPES)

print(f"\nexclusions from {len(adm):,} admissions")
print(f"  died in hospital      : {died.sum():>8,}")
print(f"  hospice/died at disch : {terminal.sum():>8,}")
print(f"  observation stays     : {observation.sum():>8,}")

coh = adm[~(died | terminal | observation)].copy()

within    = coh.days_to_next.between(0, READMIT_WINDOW_DAYS)
unplanned = ~coh.next_type.eq("ELECTIVE")
coh["readmit_30d"] = (within & unplanned).astype("int8")

print(f"  -> eligible index stays: {len(coh):>8,}  ({len(coh)/len(adm):.1%})")
print(f"\n{READMIT_WINDOW_DAYS}-day unplanned readmission rate: "
      f"{coh.readmit_30d.sum():,} / {len(coh):,} = {coh.readmit_30d.mean():.2%}")
print(f"distinct patients: {coh.subject_id.nunique():,}")

BASE_RATE = float(coh.readmit_30d.mean())
del adm; gc.collect()

## 2 &middot; Parse `omr` into tidy measurements

`omr` stores everything as text in `result_value`, so each measurement type needs its own
parser. Blood pressure arrives as `"160/70"` and must be split; weight and height are imperial
and are converted to SI so the features are interpretable.

In [ ]:
omr = pd.read_csv(tbl("omr"), usecols=["subject_id","chartdate","result_name","result_value"],
                  parse_dates=["chartdate"])
report("omr (raw)", omr)
print("\nmeasurement types:")
print(omr.result_name.value_counts().to_string())

omr = omr[omr.subject_id.isin(set(coh.subject_id))].copy()

def parse_bp(s):
    m = re.match(r"\s*(\d{2,3})\s*/\s*(\d{2,3})", str(s))
    return (float(m.group(1)), float(m.group(2))) if m else (np.nan, np.nan)

recs = []
for name, grp in omr.groupby("result_name"):
    n = str(name).lower()
    if "blood pressure" in n:
        sys_dia = grp.result_value.map(parse_bp)
        recs.append(grp.assign(measure="sbp", value=[a for a, _ in sys_dia])[["subject_id","chartdate","measure","value"]])
        recs.append(grp.assign(measure="dbp", value=[b for _, b in sys_dia])[["subject_id","chartdate","measure","value"]])
    elif "weight" in n:
        v = pd.to_numeric(grp.result_value, errors="coerce")
        if "lbs" in n:
            v = v * 0.453592          # lbs -> kg
        recs.append(grp.assign(measure="weight_kg", value=v)[["subject_id","chartdate","measure","value"]])
    elif "bmi" in n:
        recs.append(grp.assign(measure="bmi", value=pd.to_numeric(grp.result_value, errors="coerce"))
                       [["subject_id","chartdate","measure","value"]])
    elif "height" in n:
        v = pd.to_numeric(grp.result_value, errors="coerce")
        if "inches" in n:
            v = v * 2.54              # inches -> cm
        recs.append(grp.assign(measure="height_cm", value=v)[["subject_id","chartdate","measure","value"]])

obs = pd.concat(recs, ignore_index=True).dropna(subset=["value"])

# Physiologic plausibility bounds - guards against unit/typo artefacts
BOUNDS = {"sbp": (60, 260), "dbp": (30, 160), "weight_kg": (25, 350),
          "bmi": (10, 90), "height_cm": (120, 220)}
before = len(obs)
for m, (lo, hi) in BOUNDS.items():
    bad = obs.measure.eq(m) & ~obs.value.between(lo, hi)
    obs = obs[~bad]
print(f"\nimplausible values dropped: {before-len(obs):,}")
obs = downcast(obs)
report("tidy observations", obs)
print(obs.groupby("measure").value.describe()[["count","mean","50%"]].round(1).to_string())

## 3 &middot; Feasibility gate &mdash; is there a weekly signal at all?

This is the decisive section. It measures, on your full data, how long after discharge the
first observation arrives and how many distinct weeks are actually covered. **The verdict it
prints determines whether weekly re-scoring is defensible.**

In [ ]:
MONITORING_WINDOWS = (30, 90, 180)

disch = coh[["subject_id","hadm_id","dischtime","readmit_30d"]].copy()
j = disch.merge(obs, on="subject_id", how="inner")
j = j[j.chartdate > j.dischtime].copy()
j["days_after"] = (j.chartdate - j.dischtime).dt.total_seconds()/86400

if j.empty:
    raise RuntimeError("No post-discharge observations - Phase 2 is not supportable on this extract.")

first = j.groupby("hadm_id").days_after.min()
print(f"discharges with >=1 later observation : {len(first):,} / {len(disch):,} "
      f"({len(first)/len(disch):.1%})\n")
print("days until FIRST post-discharge observation")
for q in (10, 25, 50, 75, 90):
    print(f"  p{q:<3} : {np.percentile(first, q):8.0f} days")

print("\ndistinct WEEKS containing >=1 observation, per discharge")
summary = {}
for w in MONITORING_WINDOWS:
    win = j[j.days_after <= w].copy()
    if win.empty:
        print(f"  within {w:>3}d : none"); summary[w] = 0.0; continue
    win["week"] = (win.days_after // 7).astype(int)
    weeks = win.groupby("hadm_id").week.nunique().reindex(disch.hadm_id.unique()).fillna(0)
    share2 = (weeks >= 2).mean()
    summary[w] = share2
    print(f"  within {w:>3}d ({w//7:>2} possible) : median {weeks.median():.0f} "
          f"| mean {weeks.mean():.2f} | >=2 weeks {share2:.1%}")

WEEKLY_VIABLE = summary.get(30, 0) >= 0.25
print("\n" + "="*70)
if WEEKLY_VIABLE:
    print("VERDICT: enough density for weekly binning inside 30 days.")
else:
    print("VERDICT: weekly re-scoring is NOT supportable from MIMIC-IV.")
    print("  omr is opportunistic clinic measurement, not scheduled monitoring.")
    print("  Continuing with IRREGULAR-INTERVAL trend features instead, which is")
    print("  what this data can honestly support.")
print("="*70)

## 4 &middot; Irregular-interval trend features

Rather than pretending a weekly grid exists, each observation becomes a **scoring occasion**
described by:

- its value, and the change from the patient's own pre-discharge baseline,
- the **slope** across observations so far (deterioration vs recovery),
- **`days_since_last_obs`** &mdash; staleness, made an explicit feature instead of a hidden
  assumption. A model that knows an observation is 90 days old can discount it; one handed a
  forward-filled value on a fake weekly grid cannot.

In [ ]:
# Baseline: the patient's most recent measurement at or before discharge.
base = (obs.merge(disch[["subject_id","hadm_id","dischtime"]], on="subject_id")
           .query("chartdate <= dischtime")
           .sort_values("chartdate")
           .groupby(["hadm_id","measure"], as_index=False)
           .agg(baseline=("value","last")))
print(f"discharges with a pre-discharge baseline: {base.hadm_id.nunique():,}")

occ = j.sort_values(["hadm_id","measure","days_after"]).copy()
occ = occ.merge(base, on=["hadm_id","measure"], how="left")

g = occ.groupby(["hadm_id","measure"])
occ["obs_index"]          = g.cumcount() + 1
occ["prev_value"]         = g.value.shift(1)
occ["prev_days"]          = g.days_after.shift(1)
occ["days_since_last_obs"]= (occ.days_after - occ.prev_days).fillna(occ.days_after)
occ["delta_vs_prev"]      = occ.value - occ.prev_value
occ["delta_vs_baseline"]  = occ.value - occ.baseline
occ["pct_vs_baseline"]    = (occ.value - occ.baseline) / occ.baseline.replace(0, np.nan)
# points/day between consecutive observations - the interval-aware slope
occ["slope_per_day"]      = occ.delta_vs_prev / occ.days_since_last_obs.replace(0, np.nan)

report("scoring occasions", occ)
print(f"\noccasions per discharge: median "
      f"{occ.groupby('hadm_id').size().median():.0f}")
print(occ.groupby("measure")[["delta_vs_baseline","days_since_last_obs"]]
         .median().round(2).to_string())

In [ ]:
# Pivot to one row per scoring occasion (hadm_id x days_after), all measures wide.
occ["day_bin"] = occ.days_after.round().astype(int)
wide = occ.pivot_table(
    index=["hadm_id","day_bin"], columns="measure",
    values=["value","delta_vs_baseline","slope_per_day","days_since_last_obs"],
    aggfunc="last",
)
wide.columns = [f"{stat}__{m}" for stat, m in wide.columns]
wide = wide.reset_index()

# Forward-fill within a discharge so an occasion driven by one measure still
# carries the last known value of the others - with staleness recorded above.
wide = wide.sort_values(["hadm_id","day_bin"])
val_cols = [c for c in wide.columns if c.startswith(("value__","delta_vs_baseline__"))]
wide[val_cols] = wide.groupby("hadm_id")[val_cols].ffill()

wide["n_obs_so_far"] = wide.groupby("hadm_id").cumcount() + 1
wide = downcast(wide)
report("occasion-level matrix", wide)

## 5 &middot; Label &mdash; readmission *after* the scoring occasion

The Phase 1 label cannot be reused: it looks forward from discharge, whereas a monitoring score
must look forward **from the moment it is computed**. Reusing it would leak outcomes that had
already happened by the observation date.

In [ ]:
HORIZON_DAYS = 30

nxt = coh[["subject_id","hadm_id","dischtime","days_to_next","next_type"]].copy()
lab = wide.merge(nxt, on="hadm_id", how="left")

# days from THIS occasion to the next admission
lab["days_occasion_to_next"] = lab.days_to_next - lab.day_bin
lab["readmit_next_30d"] = (
    lab.days_occasion_to_next.between(0, HORIZON_DAYS)
    & ~lab.next_type.eq("ELECTIVE")
).astype("int8")

# An occasion recorded after the readmission already happened is not a prediction.
lab = lab[(lab.days_to_next.isna()) | (lab.day_bin <= lab.days_to_next)]

print(f"scoring occasions        : {len(lab):,}")
print(f"distinct discharges      : {lab.hadm_id.nunique():,}")
print(f"{HORIZON_DAYS}-day readmission rate : {lab.readmit_next_30d.mean():.2%}")
print(f"positives                : {lab.readmit_next_30d.sum():,}")

if lab.readmit_next_30d.sum() < 30:
    print("\n  !! too few positives to train on - report the feasibility finding instead")

## 6 &middot; Train the trend model

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             classification_report, precision_recall_curve)

# NOTE: `lab` already carries subject_id from the label merge above, so it is
# NOT re-merged here - doing so would produce subject_id_x / subject_id_y.

TREND_FEATURES = [c for c in wide.columns if c not in ("hadm_id",)]
TREND_FEATURES = [c for c in TREND_FEATURES if lab[c].notna().mean() > 0.02]
print(f"trend features retained (>2% coverage): {len(TREND_FEATURES)}")

yt = lab.readmit_next_30d.values
gt = lab.subject_id.values
Xt = lab[TREND_FEATURES]

if yt.sum() >= 30 and len(np.unique(gt)) >= 10:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    tr, te = next(gss.split(Xt, yt, gt))
    assert not (set(gt[tr]) & set(gt[te])), "patient leakage"

    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                       min_samples_leaf=20, l2_regularization=1.0,
                                       early_stopping=True, random_state=42)
    m.fit(Xt.iloc[tr], yt[tr])
    pt = m.predict_proba(Xt.iloc[te])[:,1]

    print(f"\n  AUC-ROC   {roc_auc_score(yt[te], pt):.3f}")
    print(f"  AUC-PR    {average_precision_score(yt[te], pt):.3f}"
          f"   (baseline {yt[te].mean():.3f})")
    print(f"  Brier     {brier_score_loss(yt[te], pt):.4f}")
    print(f"  mean pred {pt.mean():.3f} vs prevalence {yt[te].mean():.3f}")
    print()
    print(classification_report(yt[te], (pt >= 0.5).astype(int),
                                target_names=["no readmit","readmit"], digits=3,
                                zero_division=0))
else:
    print("Insufficient positives or patients to train a trend model on this extract.")
    print("This is itself the finding: MIMIC-IV does not carry enough post-discharge")
    print("density to learn a trend. Report the section 3 diagnostics and source the")
    print("weekly cadence elsewhere.")

## 7 &middot; What to carry forward

Whatever the numbers above, record these alongside them &mdash; they are the honest limits of a
Phase 2 built on MIMIC-IV:

1. **Cadence is irregular, not weekly.** Every "trend" here is measured between opportunistic
   clinic visits. Do not present it in the UI as a weekly series without saying so.
2. **Four measurements only.** Weight, BP, BMI, Height. No symptoms, no adherence, no
   engagement &mdash; the signals the feature proposal leans on hardest are absent entirely.
3. **Readmissions are under-counted.** Single-centre data: a patient readmitted to another
   hospital looks like a successful recovery here, which biases the label optimistic.
4. **Coverage is partial.** Only a minority of discharges have any post-discharge observation,
   so the trend model applies to a self-selected subgroup &mdash; those who returned to clinic
   &mdash; who are systematically different from those who did not.

To get a genuine weekly cadence, the source has to sample weekly: **All of Us** (wearables plus
surveys) or Preventra's own post-discharge telemetry once it is collecting.